# H3: base values for the pre-registration

**Question.** Before any training (PLAN.md, H3): how well does each base model already rank films by
heart size, how much can recalibration alone change its answers, and can the E arm get 1,000
training images with CTR > 0.5?

Scores are the dose run's stored P(yes), rounded to 5 decimals, so they tie where P(yes) saturates;
the tie share is reported. These values are provisional: the first H3 session re-scores the base
models on unrounded log-odds, before any training, and the bars are recomputed from the formulas
in PLAN.md. Label: segmenter CTR > 0.5. Films are resampled by patient (95% bootstrap CIs).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

ROOT = Path.cwd().parent  # the notebook runs from notebooks/
R = ROOT / "results"
models = ["4b", "1.5-4b"]
keys = ["group", "finding", "image", "level"]
measures = pd.read_csv(R / "dose_measures.csv")
scores = {m: pd.read_csv(R / f"dose_scores_{m}.csv").merge(measures.drop(columns="patient_id"), on=keys) for m in models}
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))

def auroc_patient_ci(d, score="p_yes", n_boot=1000, seed=0):
    """AUROC of score against CTR > 0.5, with a 95% CI resampling patients."""
    d = d.reset_index(drop=True)
    y = (d["ctr"] > 0.5).to_numpy()
    groups = [g.index.to_numpy() for _, g in d.groupby("patient_id")]
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(len(groups), size=len(groups))])
        if 0 < y[idx].mean() < 1:
            boots.append(roc_auc_score(y[idx], d.loc[idx, score]))
    return roc_auc_score(y, d[score]), *np.percentile(boots, [2.5, 97.5])

def tie_share(y, s):
    """Share of (positive, negative) pairs with exactly equal scores (each counts 1/2 in AUROC)."""
    t = pd.DataFrame({"y": y, "s": s}).groupby("s")["y"].agg(["sum", "size"])
    return float((t["sum"] * (t["size"] - t["sum"])).sum() / (y.sum() * (len(y) - y.sum())))

def recalibrated_brier(d):
    """Brier after refitting the model's own log-odds to the labels (logistic regression,
    5 folds by patient): what recalibration alone can do, with no new image information."""
    y, z = (d["ctr"] > 0.5).astype(int).to_numpy(), logit(d["p_yes"].to_numpy())[:, None]
    p = np.zeros(len(d))
    for train, test in GroupKFold(n_splits=5).split(z, y, d["patient_id"]):
        p[test] = LogisticRegression().fit(z[train], y[train]).predict_proba(z[test])[:, 1]
    return float(((p - y) ** 2).mean())

def within_growth_auroc(d, score="p_yes"):
    """AUROC over (positive, negative) pairs of the same growth only: ranking by CTR beyond the
    strength of the edit."""
    num = den = 0.0
    for _, g in d.groupby("level"):
        y = (g["ctr"] > 0.5).to_numpy()
        if 0 < y.sum() < len(y):
            w = y.sum() * (len(y) - y.sum())
            num, den = num + w * roc_auc_score(y, g[score]), den + w
    return num / den

## Real films and edited films, per model and phrasing

In [2]:
rows = []
for m in models:
    for phrasing in [0, 1]:
        s = scores[m][scores[m]["phrasing"] == phrasing]
        for films, d in [("real", s[s["group"] == "real"]), ("edited", s[s["group"] == "dose"])]:
            d = d.dropna(subset=["ctr"]).reset_index(drop=True)
            y = (d["ctr"] > 0.5).astype(int).to_numpy()
            auc, lo, hi = auroc_patient_ci(d)
            row = {"model": m, "phrasing": phrasing, "films": films, "n": len(d), "patients": d["patient_id"].nunique(),
                   "share_ctr_above_05": y.mean(), "ctr_auroc": auc, "ctr_auroc_ci_low": lo, "ctr_auroc_ci_high": hi,
                   "tied_pair_share": tie_share(y, d["p_yes"].to_numpy()),
                   "brier": float(((d["p_yes"] - y) ** 2).mean()), "recalibrated_brier": recalibrated_brier(d),
                   "prevalence_only_brier": y.mean() * (1 - y.mean()),
                   "median_yes_no_mass": d["yes_no_mass"].median(), "share_mass_05": (d["yes_no_mass"] >= 0.5).mean()}
            if films == "real":
                lab = d[(d["label_cardiomegaly"] == 1) | (d["no_finding"] == 1)]
                row["nih_auroc"] = roc_auc_score(lab["label_cardiomegaly"], lab["p_yes"])
                row["nih_n_cardiomegaly"], row["nih_n_no_finding"] = int(lab["label_cardiomegaly"].sum()), int(lab["no_finding"].sum())
            else:
                row["within_growth_ctr_auroc"] = within_growth_auroc(d)
                row["share_yes_below_05"] = (d.loc[y == 0, "p_yes"] > 0.5).mean()
            rows.append(row)
base = pd.DataFrame(rows)
base.round(4).T

,0,1,2,3,4,5,6,7
model,4b,4b,4b,4b,1.5-4b,1.5-4b,1.5-4b,1.5-4b
phrasing,0,0,1,1,0,0,1,1
films,real,edited,real,edited,real,edited,real,edited
n,1880,600,1880,600,1880,600,1880,600
patients,974,100,974,100,974,100,974,100
share_ctr_above_05,0.2314,0.405,0.2314,0.405,0.2314,0.405,0.2314,0.405
ctr_auroc,0.8616,0.895,0.8636,0.9392,0.8654,0.8542,0.8075,0.9231
ctr_auroc_ci_low,0.8279,0.8678,0.8329,0.9204,0.8391,0.8243,0.7604,0.9029
ctr_auroc_ci_high,0.897,0.9203,0.8977,0.9575,0.8944,0.8821,0.8552,0.9433
tied_pair_share,0.0156,0.1585,0.0167,0.0324,0.0353,0.2746,0.0082,0.0839


## Reference points

- the PadChest classifier's cardiomegaly score (not trained for CTR) on the same real films;
- the growth level alone as a score on the edited films: how far "detecting the edit strength"
  goes without reading the CTR.

In [3]:
real = measures[measures["group"] == "real"].dropna(subset=["ctr"])
edits = measures[measures["group"] == "dose"].dropna(subset=["ctr"])
lab = real[(real["label_cardiomegaly"] == 1) | (real["no_finding"] == 1)]
one = real.sample(frac=1, random_state=0).drop_duplicates("patient_id")  # one random film per patient
refs = {"padchest_ctr_auroc_real": roc_auc_score(real["ctr"] > 0.5, real["classifier_cardiomegaly"]),
        "growth_only_ctr_auroc_edited": roc_auc_score(edits["ctr"] > 0.5, edits["level"]),
        "segmenter_ctr_nih_auroc_real": roc_auc_score(lab["label_cardiomegaly"], lab["ctr"]),
        "share_ctr_above_05_one_film_per_patient": (one["ctr"] > 0.5).mean(),
        "real_patients": real["patient_id"].nunique(),
        "effusion_auroc_n_effusion": int((real["label_effusion"] == 1).sum()),
        "effusion_auroc_n_no_finding": int((real["no_finding"] == 1).sum())}
refs

{'padchest_ctr_auroc_real': 0.8774163783160324,
 'growth_only_ctr_auroc_edited': 0.8590736706205115,
 'segmenter_ctr_nih_auroc_real': 0.9168535495792892,
 'share_ctr_above_05_one_film_per_patient': np.float64(0.19917864476386038),
 'real_patients': 974,
 'effusion_auroc_n_effusion': 656,
 'effusion_auroc_n_no_finding': 802}

## Supply of E: training images with CTR > 0.5

On the 100 test dose films (same masks and prompt as E), per growth, and per source film over the
five growths E uses (0.06 to 0.30). Expected positives for N source films, with a lower bound that
counts both the uncertainty of the rate (100 films) and the film-to-film variation of the N films.

In [4]:
e = edits[edits["level"] > 0].assign(pos=lambda x: x["ctr"] > 0.5)
per_growth = e.groupby("level")["pos"].mean()
per_film = e.groupby("image")["pos"].sum()
mean, sd = per_film.mean(), per_film.std()
se = sd / np.sqrt(len(per_film))
pairs = pd.read_csv(R / "pairs_test.csv")
sham = pairs[(pairs["finding"] == "cardiomegaly") & (pairs["kind"] == "sham")]
sham_share = (sham["ctr_after"] > 0.5).mean()
supply = []
for n in [450, 500, 550]:
    expected = n * mean
    lower = expected - 1.96 * np.sqrt(n * sd**2 + n**2 * se**2)
    supply.append({"source_films": n, "radedit_runs": 6 * n, "expected_positive_additions": expected,
                   "lower_95": lower, "expected_positive_shams": n * sham_share})
print("share with CTR > 0.5 per growth:", per_growth.round(3).to_dict())
print(f"per film: mean {mean:.3f}, SD {sd:.3f}, SE {se:.3f}; shams above 0.5: {sham_share:.3f} of {len(sham)}")
supply = pd.DataFrame(supply)
supply.round(1)

share with CTR > 0.5 per growth: {0.06: 0.08, 0.12: 0.27, 0.18: 0.53, 0.24: 0.72, 0.3: 0.82}
per film: mean 2.420, SD 1.552, SE 0.155; shams above 0.5: 0.058 of 380


,source_films,radedit_runs,expected_positive_additions,lower_95,expected_positive_shams
0,450,2700,1089.0,937.7,26.1
1,500,3000,1210.0,1043.4,28.9
2,550,3300,1331.0,1149.2,31.8


## Supply with the training margin

Training examples with a segmenter CTR within 0.025 of 0.5 are dropped in every arm (the segmenter's
mean difference from CheXmask), so positives need CTR > 0.525 and negatives CTR < 0.475. E: per
growth and per source film, as above. R: on the test split (same selection rule and patient hash as
the train split), films with each label when each patient contributes at most k films drawn at
random, scaled to H3's training pool (md5 >= 0.32, 68% of patients, against 10% for the test
split).

In [5]:
e["pos_m"], e["neg_m"] = e["ctr"] > 0.525, e["ctr"] < 0.475
per_growth_m = e.groupby("level")[["pos_m", "neg_m"]].mean()
pf_m, nf_m = e.groupby("image")["pos_m"].sum(), e.groupby("image")["neg_m"].sum()
mean_m, sd_m = pf_m.mean(), pf_m.std()
se_m = sd_m / np.sqrt(len(pf_m))
sham_m = {"sham_share_ctr_above_0525": (sham["ctr_after"] > 0.525).mean(), "sham_share_ctr_below_0475": (sham["ctr_after"] < 0.475).mean()}
supply_m = []
for n in [600, 700, 750, 800]:
    expected = n * mean_m
    supply_m.append({"source_films": n, "radedit_runs": 6 * n, "expected_positives": expected,
                     "lower_95": expected - 1.96 * np.sqrt(n * sd_m**2 + n**2 * se_m**2),
                     "expected_negatives": n * (nf_m.mean() + sham_m["sham_share_ctr_below_0475"])})
supply_m = pd.DataFrame(supply_m)
scale = 0.68 / 0.10
r_supply = []
for cap in [1, 2, 3]:
    draws = []
    for seed in range(5):  # average over 5 random draws of the films kept per patient
        d = real.sample(frac=1, random_state=seed).groupby("patient_id").head(cap)
        draws.append(((d["ctr"] > 0.525).sum(), (d["ctr"] < 0.475).sum()))
    pos, neg = np.mean(draws, axis=0)
    r_supply.append({"films_per_patient_at_most": cap, "test_positives": pos, "pool_positives": pos * scale,
                     "test_negatives": neg, "pool_negatives": neg * scale})
r_supply = pd.DataFrame(r_supply)
print("E, share per growth (CTR > 0.525 / < 0.475):", per_growth_m.round(3).to_dict())
print(f"E, positives per film with the margin: mean {mean_m:.3f}, SD {sd_m:.3f}, SE {se_m:.3f}; negatives per film {nf_m.mean():.3f};", {k: round(v, 3) for k, v in sham_m.items()})
display(supply_m.round(1))
r_supply.round(0)

E, share per growth (CTR > 0.525 / < 0.475): {'pos_m': {0.06: 0.01, 0.12: 0.1, 0.18: 0.31, 0.24: 0.55, 0.3: 0.7}, 'neg_m': {0.06: 0.7, 0.12: 0.43, 0.18: 0.28, 0.24: 0.17, 0.3: 0.1}}
E, positives per film with the margin: mean 1.670, SD 1.386, SE 0.139; negatives per film 1.680; {'sham_share_ctr_above_0525': np.float64(0.018), 'sham_share_ctr_below_0475': np.float64(0.847)}


,source_films,radedit_runs,expected_positives,lower_95,expected_negatives
0,600,3600,1002.0,826.0,1516.4
1,700,4200,1169.0,965.7,1769.2
2,750,4500,1252.5,1035.6,1895.5
3,800,4800,1336.0,1105.5,2021.9


,films_per_patient_at_most,test_positives,pool_positives,test_negatives,pool_negatives
0,1,111.0,755.0,655.0,4455.0
1,2,152.0,1036.0,824.0,5605.0
2,3,177.0,1206.0,921.0,6266.0


In [6]:
out = [{"model": r["model"], "phrasing": r["phrasing"], "films": r["films"], "metric": k, "value": v}
       for r in rows for k, v in r.items() if k not in ("model", "phrasing", "films")]
out += [{"model": "", "phrasing": "", "films": "", "metric": k, "value": v} for k, v in refs.items()]
out += [{"model": "", "phrasing": "", "films": "edited", "metric": f"share_ctr_above_05_growth_{g:.2f}", "value": v}
        for g, v in per_growth.items()]
out += [{"model": "", "phrasing": "", "films": "edited", "metric": k, "value": v} for k, v in
        [("positives_per_film_mean", mean), ("positives_per_film_sd", sd), ("positives_per_film_se", se),
         ("sham_share_ctr_above_05", sham_share)]]
out += [{"model": "", "phrasing": "", "films": "E supply", "metric": f"{k}_{int(r['source_films'])}_films", "value": r[k]}
        for _, r in supply.iterrows() for k in ["expected_positive_additions", "lower_95", "expected_positive_shams"]]
out += [{"model": "", "phrasing": "", "films": "edited", "metric": f"share_ctr_above_0525_growth_{g:.2f}", "value": v}
        for g, v in per_growth_m["pos_m"].items()]
out += [{"model": "", "phrasing": "", "films": "edited", "metric": k, "value": v} for k, v in
        [("positives_per_film_margin_mean", mean_m), ("positives_per_film_margin_sd", sd_m),
         ("positives_per_film_margin_se", se_m), ("negatives_per_film_margin_mean", nf_m.mean())] + list(sham_m.items())]
out += [{"model": "", "phrasing": "", "films": "E supply, margin", "metric": f"{k}_{int(r['source_films'])}_films", "value": r[k]}
        for _, r in supply_m.iterrows() for k in ["expected_positives", "lower_95", "expected_negatives"]]
out += [{"model": "", "phrasing": "", "films": "R supply, margin", "metric": f"{k}_at_most_{int(r['films_per_patient_at_most'])}",
         "value": r[k]} for _, r in r_supply.iterrows() for k in ["pool_positives", "pool_negatives"]]
pd.DataFrame(out).assign(value=lambda x: x["value"].astype(float).round(4)).to_csv(R / "h3_base.csv", index=False)
print("written results/h3_base.csv,", len(out), "rows")

written results/h3_base.csv, 170 rows


## Takeaways

In [7]:
for _, r in base[base["phrasing"] == 0].iterrows():
    print(f"MedGemma {r['model']}, {r['films']} films: CTR-AUROC {r['ctr_auroc']:.3f} ({r['ctr_auroc_ci_low']:.3f}-"
          f"{r['ctr_auroc_ci_high']:.3f}), tied pairs {r['tied_pair_share']:.1%}; Brier {r['brier']:.3f}, "
          f"recalibrated {r['recalibrated_brier']:.3f}; median yes/no mass {r['median_yes_no_mass']:.2f}.")
print(f"E supply: {mean:.2f} positives per source film (SE {se:.2f}); 500 films give {supply.iloc[1]['expected_positive_additions']:.0f} "
      f"expected (lower 95% {supply.iloc[1]['lower_95']:.0f}).")

MedGemma 4b, real films: CTR-AUROC 0.862 (0.828-0.897), tied pairs 1.6%; Brier 0.196, recalibrated 0.124; median yes/no mass 0.97.
MedGemma 4b, edited films: CTR-AUROC 0.895 (0.868-0.920), tied pairs 15.8%; Brier 0.332, recalibrated 0.113; median yes/no mass 1.00.
MedGemma 1.5-4b, real films: CTR-AUROC 0.865 (0.839-0.894), tied pairs 3.5%; Brier 0.248, recalibrated 0.120; median yes/no mass 0.06.
MedGemma 1.5-4b, edited films: CTR-AUROC 0.854 (0.824-0.882), tied pairs 27.5%; Brier 0.393, recalibrated 0.128; median yes/no mass 0.81.
E supply: 2.42 positives per source film (SE 0.16); 500 films give 1210 expected (lower 95% 1043).
